In [ ]:
# Cell 1: Imports & Reproducibility
import os
import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [ ]:
ANOMALY_DIR    = './anomaly'        # flat folder with all 13 prefixed
FOUR_CLASS_DIR = './four_class'

BATCH_SIZE      = 8
EPOCHS          = 30
CLIP_LEN        = 16
IMG_HEIGHT      = 64
IMG_WIDTH       = 64
NUM_CLASSES     = 4
STEPS_PER_EPOCH = 50
VAL_STEPS       = 10


In [ ]:
# Define super‐class mapping
mapping = {
    'Abuse':    'Violent_Crimes',
    'Assault':  'Violent_Crimes',
    'Fighting': 'Violent_Crimes',
    'Shooting': 'Violent_Crimes',
    'Robbery':  'Property_Theft',
    'Burglary': 'Property_Theft',
    'Shoplifting': 'Property_Theft',
    'Stealing':    'Property_Theft',
    'Arson':       'Destructive_Crimes',
    'Explosion':   'Destructive_Crimes',
    'Vandalism':   'Destructive_Crimes',
    'Arrest':        'Legal_Accidents',
    'RoadAccidents': 'Legal_Accidents'
}

In [ ]:


# Create a CLAHE object: clipLimit=2.0, tileGridSize=8×8
_CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

def apply_clahe_rgb(frame: np.ndarray) -> np.ndarray:
    """
    Input: frame in RGB [0,1], shape (H, W, 3), dtype float32.
    Output: same shape/dtype, with CLAHE on the L channel.
    """
    # convert 0–1 float → 0–255 uint8
    img = (frame * 255).astype(np.uint8)
    # RGB → LAB
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    # apply CLAHE on L
    l_eq = _CLAHE.apply(l)
    lab_eq = cv2.merge([l_eq, a, b])
    # back to RGB, float32 [0,1]
    rgb_eq = cv2.cvtColor(lab_eq, cv2.COLOR_LAB2RGB).astype(np.float32) / 255.0
    return rgb_eq


In [ ]:
def video_clip_generator(normal_dir,
                         anomaly_dir,
                         batch_size,
                         clip_len,
                         img_h,
                         img_w,
                         shuffle=True):

    # 1) gather all video paths + labels
    vids, labels = [], []
    for fname in os.listdir(normal_dir):
        if fname.lower().endswith(('.mp4','avi','mov')):
            vids.append(os.path.join(normal_dir, fname))
            labels.append(0)
    for fname in os.listdir(anomaly_dir):
        if fname.lower().endswith(('.mp4','avi','mov')):
            vids.append(os.path.join(anomaly_dir, fname))
            labels.append(1)

    idxs = np.arange(len(vids))

    # 2) infinite loop
    while True:
        if shuffle:
            np.random.shuffle(idxs)
        # 3) batch slices
        for start in range(0, len(idxs), batch_size):
            batch_i = idxs[start:start+batch_size]
            X = np.zeros((len(batch_i), clip_len, img_h, img_w, 3), dtype=np.float32)
            y = np.zeros((len(batch_i),), dtype=int)

            # 4) fill each sample
            for i, j in enumerate(batch_i):
                cap = cv2.VideoCapture(vids[j])
                frames = []
                total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                for f in np.linspace(0, total-1, clip_len, dtype=int):
                    cap.set(cv2.CAP_PROP_POS_FRAMES, f)
                    ret, frame = cap.read()
                    if not (ret and frame is not None):
                        break
                    # resize & convert BGR→RGB
                    frame = cv2.resize(frame, (img_w, img_h))
                    frame_rgb = frame[..., ::-1].astype(np.float32) / 255.0
                    # apply CLAHE on the luminance channel
                    frame_eq = apply_clahe_rgb(frame_rgb)
                    frames.append(frame_eq)
                cap.release()

                if len(frames) == clip_len:
                    X[i] = np.stack(frames)
                else:
                    X[i] = np.random.rand(clip_len, img_h, img_w, 3)

                y[i] = labels[j]

            # 5) to one-hot
            yield X, tf.keras.utils.to_categorical(y, NUM_CLASSES)

In [ ]:
# Cell 5: Build Hybrid CNN–BiLSTM for 4 Classes
def residual_block(x, filters, kernel=3):
    skip = x
    x = layers.TimeDistributed(layers.Conv2D(filters, kernel, padding='same', activation='relu'))(x)
    x = layers.TimeDistributed(layers.Conv2D(filters, kernel, padding='same'))(x)
    x = layers.add([x, skip])
    x = layers.TimeDistributed(layers.Activation('relu'))(x)
    x = layers.TimeDistributed(layers.MaxPooling2D())(x)
    return x

def build_model_mc(input_shape, n_classes):
    inp = layers.Input(shape=input_shape)
    x = layers.TimeDistributed(layers.Conv2D(32,3,padding='same',activation='relu'))(inp)
    x = layers.TimeDistributed(layers.MaxPooling2D())(x)
    x = residual_block(x, 32)
    x = layers.TimeDistributed(layers.Conv2D(64,3,padding='same',activation='relu'))(x)
    x = layers.TimeDistributed(layers.MaxPooling2D())(x)
    x = layers.TimeDistributed(layers.Flatten())(x)
    x = layers.SpatialDropout1D(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=False))(x)
    out = layers.Dense(n_classes, activation='softmax')(x)
    return models.Model(inp, out)

classes = sorted(set(mapping.values()))
model_mc = build_model_mc((CLIP_LEN, IMG_HEIGHT, IMG_WIDTH, 3), NUM_CLASSES)
model_mc.compile(optimizer=optimizers.Adam(1e-4),
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])
model_mc.summary()


In [ ]:
# Cell 6: Callbacks
es_cb    = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
ckpt_cb  = callbacks.ModelCheckpoint('best_task2.h5', monitor='val_accuracy', save_best_only=True)
rlrop_cb = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)


In [ ]:
# Cell 7: Train on Real Data
train_gen = video_clip_generator(FOUR_CLASS_DIR, classes,
                                   BATCH_SIZE, CLIP_LEN,
                                   IMG_HEIGHT, IMG_WIDTH,
                                   shuffle=True)
val_gen   = video_clip_generator(FOUR_CLASS_DIR, classes,
                                   BATCH_SIZE, CLIP_LEN,
                                   IMG_HEIGHT, IMG_WIDTH,
                                   shuffle=False)

history_mc = model_mc.fit(
    train_gen,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_gen,
    validation_steps=VAL_STEPS,
    callbacks=[es_cb, ckpt_cb, rlrop_cb],
    verbose=1
)


In [ ]:
# Cell 8: Plot Training & Validation Curves
plt.figure(figsize=(14,5))

plt.subplot(1,2,1)
plt.plot(history_mc.history['accuracy'],    label='Train Acc')
plt.plot(history_mc.history['val_accuracy'],label='Val Acc')
plt.axvline(23, linestyle='--', color='gray', label='Early Stop')
plt.title('Accuracy vs Epoch')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()

plt.subplot(1,2,2)
plt.plot(history_mc.history['loss'],       label='Train Loss')
plt.plot(history_mc.history['val_loss'],   label='Val Loss')
plt.axvline(23, linestyle='--', color='gray')
plt.title('Loss vs Epoch')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()

plt.tight_layout(); plt.show()


In [ ]:
def single_clip_gen(vid_paths, clip_len, img_h, img_w):
    for path in vid_paths:
        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frames = []
        for f in np.linspace(0, total-1, clip_len, dtype=int):
            cap.set(cv2.CAP_PROP_POS_FRAMES, f)
            ret, frame = cap.read()
            if not ret: break
            frame = cv2.resize(frame, (img_w, img_h))
            frames.append(frame[..., ::-1]/255.0)
        cap.release()
        if len(frames)<clip_len:
            pad = np.stack([frames[-1]]*(clip_len-len(frames))) if frames else np.random.rand(clip_len,img_h,img_w,3)
            frames += list(pad)
        yield np.expand_dims(np.stack(frames), 0)

# gather test videos
normal_test  = sorted(glob.glob('./test/normal/*.mp4'))
anomaly_test = sorted(glob.glob('./test/anomaly/*.mp4'))
test_vids    = normal_test + anomaly_test
test_labels  = [0]*len(normal_test) + [1]*len(anomaly_test)

# inference
y_true, y_pred = [], []
for path, lbl in zip(test_vids, test_labels):
    clip = next(single_clip_gen([path], CLIP_LEN, IMG_HEIGHT, IMG_WIDTH))
    pred = np.argmax(model_mc.predict(clip), axis=1)[0]
    y_true.append(lbl)
    y_pred.append(pred)

print("Classification Report:")
print(classification_report(y_true, y_pred, digits=4, target_names=classes))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.show()

acc = np.mean(np.array(y_true)==np.array(y_pred))
print(f"Final Simulated Test Accuracy: {acc*100:.3f}%")
